# 财会数据分析 · 交互探索

先确保已激活项目环境（本 notebook 的 kernel 指向 `.venv`）。下面单元格依次运行即可。

演示数据与 akshare 拉取的真实财报**结构完全相同**（「报告日 × 科目」宽表，新浪科目口径，单位元），
共用 `src/io_utils.py` 的加载与勾稽逻辑，没有字段适配层。演示数据为虚构公司，仅用于教学。

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.io_utils import load_all, check_balance_identity, check_cash_ties
from src import ratios
print('路径:', ROOT)

路径: /Users/pengjiakun/pythonworkspace/finance-starter


## 1. 加载数据并做勾稽校验

In [2]:
inc, bs, cf = load_all()
print('利润表'); display(inc)
check_balance_identity(bs)

利润表


年份,2022,2023,2024
项目,,,
报告日,2.022123e+07,2.023123e+07,2.024123e+07
营业总收入,1.000000e+09,1.150000e+09,1.280000e+09
营业收入,1.000000e+09,1.150000e+09,1.280000e+09
营业成本,6.500000e+08,7.475000e+08,8.240000e+08
营业税金及附加,8.000000e+06,9.200000e+06,1.024000e+07
销售费用,8.000000e+07,9.200000e+07,1.024000e+08
管理费用,6.000000e+07,6.900000e+07,7.680000e+07
研发费用,5.000000e+07,5.750000e+07,6.400000e+07
财务费用,1.000000e+07,1.150000e+07,1.280000e+07


,资产总计,负债+权益,差额(应为0),是否平衡
年份,,,,
2022,1.050000e+09,1.050000e+09,0.0,True
2023,1.175000e+09,1.175000e+09,0.0,True
2024,1.300000e+09,1.300000e+09,0.0,True


## 2. 查看盈利能力

In [3]:
summary = ratios.summary(inc, bs, cf)
display(summary['盈利能力'].round(4))

,毛利率,营业利润率,净利率,ROE(净资产收益率),ROA(总资产收益率)
年份,,,,,
2022,0.3500,0.1920,0.1425,0.2127,0.1357
2023,0.3500,0.1942,0.1450,0.2194,0.1419
2024,0.3562,0.1951,0.1469,0.2187,0.1447


## 3. 杜邦分解：ROE = 净利率 × 总资产周转率 × 权益乘数

In [4]:
display(summary['杜邦分解'].round(4))

,净利率,总资产周转率,权益乘数,ROE(分解乘积),ROE(直接计算)
年份,,,,,
2022,0.1425,0.9524,1.5672,0.2127,0.2127
2023,0.1450,0.9787,1.5461,0.2194,0.2194
2024,0.1469,0.9846,1.5116,0.2187,0.2187


## 4. 接入 A 股真实财报（akshare）

前面用的是演示数据。真实财报与演示数据**结构完全相同**，因此直接用同一个 `load_all(source='real', code=...)` 加载，无需任何字段转换：

1. **拉取**：`scripts/fetch_akshare.py 600519` 把贵州茅台三大报表存到 `data/raw/`（本项目已预置 600519 与 000001 两只）。
2. **加载**：`load_all(source='real', code='600519')` 读「报告日 × 科目」宽表，筛年报并转成「科目 × 年份」结构。
3. **分析**：结构与演示数据一致，直接复用前面的 `ratios.summary()` 等函数。

数据源是新浪财经免费接口，可能变动或限流。真实财报里「期末现金及现金等价物」与「货币资金」口径不同（后者含受限资金），两者不相等是正常现象，不代表报表有误。

### 4.1 查看已拉取的真实数据文件

In [5]:
for f in sorted((ROOT / 'data' / 'raw').glob('*.csv')):
    print(f.name)

000001_balance.csv
000001_cashflow.csv
000001_income.csv
600519_balance.csv
600519_cashflow.csv
600519_income.csv


### 4.2 用统一加载函数读取，并做勾稽校验

`load_all(source='real', code=...)` 与演示数据走同一个加载函数。资产 = 负债 + 权益 的勾稽在真实数据上同样成立。

In [6]:
r_inc, r_bs, r_cf = load_all(source='real', code='600519')
print('利润表：', r_inc.shape, '（科目 × 年份）')
display(r_inc.iloc[:, -3:].round(0))
print('资产负债表勾稽（近 3 年）：')
display(check_balance_identity(r_bs).tail(3))

利润表： (88, 28) （科目 × 年份）


年份,2023,2024,2025
项目,,,
报告日,2.023123e+07,2.024123e+07,2.025123e+07
营业总收入,1.505603e+11,1.741441e+11,1.720542e+11
营业收入,1.476936e+11,1.708992e+11,1.688381e+11
利息收入,2.866725e+09,3.244918e+09,3.216069e+09
已赚保费,NaN,NaN,NaN
...,...,...,...
负债合计,NaN,NaN,NaN
所有者权益(或股东权益)合计,NaN,NaN,NaN
负债和所有者权益(或股东权益)总计,NaN,NaN,NaN


资产负债表勾稽（近 3 年）：


,资产总计,负债+权益,差额(应为0),是否平衡
年份,,,,
2023,2.726997e+11,2.726997e+11,0.0,True
2024,2.989446e+11,2.989446e+11,0.0,True
2025,3.038348e+11,3.038348e+11,0.0,True


### 4.3 用同一套比率模块分析真实财报

加载后的三张表直接传给 `ratios.summary()`，无需改动分析代码。下面看最近 3 年的盈利能力与杜邦分解——茅台净利率约 52%、ROE 约 34%，量级与公开财报一致。

In [7]:
r_summary = ratios.summary(r_inc, r_bs, r_cf)
print('盈利能力（近 3 年）：')
display(r_summary['盈利能力'].tail(3).round(4))
print('杜邦分解（近 3 年）：')
display(r_summary['杜邦分解'].tail(3).round(4))

盈利能力（近 3 年）：


,毛利率,营业利润率,净利率,ROE(净资产收益率),ROA(总资产收益率)
年份,,,,,
2023,0.9196,0.7022,0.5249,0.3466,0.2843
2024,0.9193,0.7003,0.5227,0.3691,0.2988
2025,0.9118,0.6800,0.5053,0.3359,0.2808


杜邦分解（近 3 年）：


,净利率,总资产周转率,权益乘数,ROE(分解乘积),ROE(直接计算)
年份,,,,,
2023,0.5249,0.5416,1.2193,0.3466,0.3466
2024,0.5227,0.5717,1.2353,0.3691,0.3691
2025,0.5053,0.5557,1.1964,0.3359,0.3359


### 4.4 拉取其他股票

在项目根目录执行，把代码换成任意 6 位 A 股代码即可（6 开头按沪市 sh，其余按深市 sz 处理）：

```bash
.venv/bin/python scripts/fetch_akshare.py 000001
```

脚本会把三大报表写入 `data/raw/{代码}_*.csv`。拉完后把上面 4.2 里 `code='600519'` 换成新代码，整段分析就切换到新公司。

真实报表科目比演示数据多得多；`load_all()` 保留全部科目，分析模块只取其中比率所需的部分，其余科目可在 `data/raw/` 原始 CSV 中做自定义分析。